# `ablation_configs.py` — Ablation & Baseline Experiment Configurations

## Purpose
Generates **config overrides** for each ablation study variant. Instead of duplicating config YAML files, each function takes the base config and returns a **deep copy** with only the relevant parameters changed.

## Design Pattern
Every function returns a list of `(experiment_id, description, config_dict)` tuples — called `ExperimentSpec`. The `experiment_runner.py` iterates these to train and evaluate each variant.

## Ablation Studies

| ID | What's being tested | Hypothesis |
|----|---------------------|------------|
| A1 | GCN on/off | Does the dependency graph improve sentiment separation? |
| A2 | Aspect attention vs CLS pooling | Does aspect-guided attention outperform a generic CLS vector? |
| A3 | Loss function variants | Which imbalance-handling loss works best? |
| A4 | With/without synthetic augmentation | Does LLM-generated data help rare classes? |
| A5 | Aspect-specific vs shared classifier head | Do dedicated heads per aspect outperform a shared head? |
| A6 | MSR evaluation with/without GCN | Does the GCN specifically help Mixed Sentiment Resolution? |
| A7 | Hybrid loss weight tuning | What's the best Focal:CB ratio? |

## Baselines (B series)
- B1: Plain RoBERTa (no aspect awareness)
- B2: Full model + vanilla Cross-Entropy
- B3: BERT-base (instead of RoBERTa)
- B4: TF-IDF + LinearSVC (classical ML)
- B5: Flat ABSA — prior-art-style baseline (no GCN, shared head, CE loss)

In [ ]:
import copy
from typing import Dict, List, Tuple

# Type alias: each experiment is a (id, description, config_override) triple
ExperimentSpec = Tuple[str, str, dict]

## Aggregator Function

In [ ]:
def get_all_ablation_specs(base_config: dict) -> List[ExperimentSpec]:
    """
    Returns ALL ablation experiment specs by calling each A-series function.
    Pass the base config (loaded from config.yaml) to get fully-formed configs.
    """
    specs = []
    specs.extend(ablation_1_gcn(base_config))
    specs.extend(ablation_2_aspect_attention(base_config))
    specs.extend(ablation_3_loss_function(base_config))
    specs.extend(ablation_4_augmentation(base_config))
    specs.extend(ablation_5_classifier_head(base_config))
    specs.extend(ablation_6_mixed_sentiment(base_config))
    specs.extend(ablation_7_hybrid_weights(base_config))
    return specs

## A1: GCN On/Off

In [ ]:
def ablation_1_gcn(base_config: dict) -> List[ExperimentSpec]:
    """
    Tests whether the Dependency GCN improves performance.
    Hypothesis: GCN captures syntactic relationships critical for MSR.
    """
    full   = copy.deepcopy(base_config)                  # Deep copy prevents mutating base_config
    full['experiment']['name'] = 'A1_full_model'         # Add GCN (default)

    no_gcn = copy.deepcopy(base_config)
    no_gcn['model']['use_dependency_gcn']    = False     # Disable GCN module
    no_gcn['data']['use_dependency_parsing'] = False     # Also skip parsing (saves compute)
    no_gcn['experiment']['name']             = 'A1_no_gcn'

    return [
        ('A1_full_model', 'Full model (with Dependency GCN)', full),
        ('A1_no_gcn',     'No GCN — aspect attention only',   no_gcn),
    ]

## A2: Aspect Attention vs CLS Pooling

In [ ]:
def ablation_2_aspect_attention(base_config: dict) -> List[ExperimentSpec]:
    attention = copy.deepcopy(base_config)
    attention['experiment']['name'] = 'A2_aspect_attention'       # Aspect-guided MHA (default)

    cls_only = copy.deepcopy(base_config)
    cls_only['model']['use_aspect_attention'] = False             # Fall back to CLS token
    cls_only['experiment']['name']            = 'A2_cls_pooling'

    return [
        ('A2_aspect_attention', 'Aspect-guided MHA attention',     attention),
        ('A2_cls_pooling',      'CLS token pooling (no attention)', cls_only),
    ]

## A3: Loss Function Variants

In [ ]:
def ablation_3_loss_function(base_config: dict) -> List[ExperimentSpec]:
    """
    Tests each loss component individually and in combination to identify
    which ones contribute to handling class imbalance.
    """
    def make_cfg(name, weights):
        cfg = copy.deepcopy(base_config)
        cfg['training']['loss_weights'] = weights  # Controls Focal/CB/Dice weighting
        cfg['experiment']['name']       = name
        return cfg

    hybrid     = make_cfg('A3_hybrid_loss',  {'focal': 1.0, 'cb': 0.5, 'dice': 0.3})  # All 3
    focal_only = make_cfg('A3_focal_only',   {'focal': 1.0, 'cb': 0.0, 'dice': 0.0})  # Only focal
    cb_only    = make_cfg('A3_cb_only',      {'focal': 0.0, 'cb': 1.0, 'dice': 0.0})  # Only CB
    dice_only  = make_cfg('A3_dice_only',    {'focal': 0.0, 'cb': 0.0, 'dice': 1.0})  # Only dice

    ce = copy.deepcopy(base_config)
    ce['training']['use_ce_loss'] = True   # Flag picked up by experiment_runner to use CrossEntropyLossWrapper
    ce['experiment']['name']      = 'A3_ce_loss'

    return [
        ('A3_hybrid_loss', 'Hybrid Loss (Focal + CB + Dice)', hybrid),
        ('A3_focal_only',  'Focal Loss only',                 focal_only),
        ('A3_cb_only',     'Class-Balanced Loss only',        cb_only),
        ('A3_dice_only',   'Dice Loss only',                  dice_only),
        ('A3_ce_loss',     'Cross-Entropy (no imbalance)',    ce),
    ]

## A4–A7: Augmentation, Classifier Head, MSR, Hybrid Weights

In [ ]:
def ablation_4_augmentation(base_config):
    """Tests whether LLM-generated synthetic data improves rare-class performance."""
    with_aug = copy.deepcopy(base_config)
    with_aug['data']['train_path'] = 'data/splits/train_augmented.csv'  # Uses augmented set
    with_aug['experiment']['name'] = 'A4_with_augmentation'

    no_aug = copy.deepcopy(base_config)
    no_aug['data']['train_path'] = 'data/splits/train.csv'              # Uses original set
    no_aug['experiment']['name'] = 'A4_no_augmentation'

    return [('A4_with_augmentation', 'With LLM augmentation (10,050 samples)', with_aug),
            ('A4_no_augmentation',   'Without augmentation (9,240 samples)',   no_aug)]


def ablation_5_classifier_head(base_config):
    """Tests 7 aspect-specific heads vs 1 shared head."""
    aspect_specific = copy.deepcopy(base_config)
    aspect_specific['model']['use_shared_classifier'] = False  # Per-aspect heads (default)
    aspect_specific['experiment']['name'] = 'A5_aspect_specific_heads'

    shared = copy.deepcopy(base_config)
    shared['model']['use_shared_classifier'] = True           # One shared head
    shared['experiment']['name'] = 'A5_shared_head'

    return [('A5_aspect_specific_heads', '7 aspect-specific classifier heads', aspect_specific),
            ('A5_shared_head',           'Single shared classifier head',       shared)]


def ablation_6_mixed_sentiment(base_config):
    """
    Both variants use the full MSR evaluator so we can directly compare
    how much the GCN specifically helps handle conflicting sentiments within a review.
    """
    with_gcn = copy.deepcopy(base_config)
    with_gcn['experiment']['name']         = 'A6_msr_with_gcn'
    with_gcn['experiment']['evaluate_msr'] = True  # Trigger MixedSentimentEvaluator post-test

    no_gcn = copy.deepcopy(base_config)
    no_gcn['model']['use_dependency_gcn']    = False
    no_gcn['data']['use_dependency_parsing'] = False
    no_gcn['experiment']['name']             = 'A6_msr_no_gcn'
    no_gcn['experiment']['evaluate_msr']     = True

    return [('A6_msr_with_gcn', 'MSR Eval: Full model + GCN', with_gcn),
            ('A6_msr_no_gcn',   'MSR Eval: No GCN (attention only)', no_gcn)]


def ablation_7_hybrid_weights(base_config):
    """Tunes the Focal:CB ratio without Dice (Dice was dropped after A3)."""
    def make_cfg(name, weights):
        cfg = copy.deepcopy(base_config)
        cfg['training']['loss_weights'] = weights
        cfg['experiment']['name']       = name
        return cfg

    return [
        ('A7_hybrid_cb_05', 'Hybrid: Focal 1.0 + CB 0.5 (no Dice)', make_cfg('A7_hybrid_cb_05', {'focal': 1.0, 'cb': 0.5, 'dice': 0.0})),
        ('A7_hybrid_cb_10', 'Hybrid: Focal 1.0 + CB 1.0 (no Dice)', make_cfg('A7_hybrid_cb_10', {'focal': 1.0, 'cb': 1.0, 'dice': 0.0})),
    ]